In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

base_path = "/content/drive/MyDrive/Corpus-LST20/LST20_Corpus"
print(os.listdir(base_path))

['._eval', '._train', '._test', 'genres.txt', '._genres.txt', '._LST20 Brief Specification.pdf', 'LST20 Brief Specification.pdf', '.DS_Store', '._.DS_Store', '._LST20 Annotation Guideline.pdf', 'LST20 Annotation Guideline.pdf', '._AGREEMENT.txt', 'AGREEMENT.txt', 'DESCRIPTION.txt', '._DESCRIPTION.txt', 'train', 'test', 'eval']


In [3]:
import os

def read_lst20(folder):
    texts = []
    labels = []

    for file in os.listdir(folder):
        if file.startswith("._"):
            continue

        path = os.path.join(folder, file)

        with open(path, encoding="utf-8") as f:
            chars = []
            tags = []

            for line in f:
                line = line.strip()

                if not line:
                    if chars:
                        texts.append(chars)
                        labels.append(tags)
                        chars, tags = [], []
                    continue

                parts = line.split()
                char = parts[0]
                tag = parts[-1][0]  # B/I/E

                chars.append(char)
                tags.append(tag)

    return texts, labels

In [4]:
train_path = "/content/drive/MyDrive/Corpus-LST20/LST20_Corpus/train"

texts, labels = read_lst20(train_path)

print(texts[0][:20])
print(labels[0][:20])

['สิงโต', 'ระอุ', 'แบร์รี', 'ปะทะ', 'เคฮิลล์']
['B', 'I', 'I', 'I', 'E']


In [5]:
def convert_to_char_level(texts, labels):
    new_texts = []
    new_labels = []

    for words, tags in zip(texts, labels):
        chars = []
        char_tags = []

        for word in words:
            if len(word) == 1:
                # 🔥 กรณีตัวเดียว → ให้เป็น E หรือ I (สำคัญ)
                chars.append(word)
                char_tags.append("E")   # หรือ "I" ก็ได้ แต่ห้ามเป็น B เยอะ
            else:
                for i, c in enumerate(word):
                    chars.append(c)

                    if i == 0:
                        char_tags.append("B")
                    elif i == len(word) - 1:
                        char_tags.append("E")
                    else:
                        char_tags.append("I")

        new_texts.append(chars)
        new_labels.append(char_tags)

    return new_texts, new_labels

In [6]:
texts_char, labels_char = convert_to_char_level(texts, labels)

print(texts_char[0][:20])
print(labels_char[0][:20])

['ส', 'ิ', 'ง', 'โ', 'ต', 'ร', 'ะ', 'อ', 'ุ', 'แ', 'บ', 'ร', '์', 'ร', 'ี', 'ป', 'ะ', 'ท', 'ะ', 'เ']
['B', 'I', 'I', 'I', 'E', 'B', 'I', 'I', 'E', 'B', 'I', 'I', 'I', 'I', 'E', 'B', 'I', 'I', 'E', 'B']


In [7]:
from collections import Counter

# vocab
all_chars = [c for sent in texts_char for c in sent]
char2id = {c:i+1 for i,c in enumerate(set(all_chars))}
char2id["<PAD>"] = 0

label2id = {"B":0, "I":1, "E":2}
id2label = {v:k for k,v in label2id.items()}

In [8]:
def encode(texts, labels):
    X, y = [], []

    for t, l in zip(texts, labels):
        X.append([char2id[c] for c in t])
        y.append([label2id[tag] for tag in l])

    return X, y

X, y = encode(texts_char, labels_char)

In [9]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

In [12]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    X_batch, y_batch = zip(*batch)

    X_pad = pad_sequence(X_batch, batch_first=True, padding_value=0)  # char pad
    y_pad = pad_sequence(y_batch, batch_first=True, padding_value=0)  # label pad

    return X_pad, y_pad

In [13]:
dataset = MyDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=32,   # สำคัญมาก
    shuffle=True,
    collate_fn=collate_fn
)

In [14]:
import torch.nn as nn

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, 4)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x)
        return x

In [ ]:
import torch
import torch.nn as nn

model = BiLSTM(len(char2id))

loss_fn = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()

    total_loss = 0
    step = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        # 🔍 DEBUG: shape
        if step == 0:
            print("X_batch:", X_batch.shape)
            print("y_batch:", y_batch.shape)

        outputs = model(X_batch)

        # 🔍 DEBUG: output shape
        if step == 0:
            print("outputs:", outputs.shape)

        loss = loss_fn(
            outputs.view(-1, 4),
            y_batch.view(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        step += 1

    print(f"epoch {epoch+1} | loss: {total_loss/step:.4f}")

X_batch: torch.Size([32, 431])
y_batch: torch.Size([32, 431])
outputs: torch.Size([32, 431, 4])


In [ ]:
from collections import Counter

all_labels = []

for X_batch, y_batch in loader:
    all_labels += y_batch.view(-1).tolist()

print(Counter(all_labels))

In [ ]:
with open("/content/ws_test.txt", encoding="utf-8") as f:
    text = f.read()

# 🔥 ใช้แบบนี้เท่านั้น
chars = [c for c in text if c != " " and c != "\n"]

print(len(chars))  # ต้อง 35182

In [ ]:
print(chars)

In [ ]:
x = [char2id.get(c, char2id.get("<UNK>", 0)) for c in chars]
x = torch.tensor(x).unsqueeze(0)

model.eval()
with torch.no_grad():
    outputs = model(x)
    pred_ids = outputs.argmax(dim=-1)[0]

In [ ]:
label2id = {
    "<PAD>": 0,
    "B": 1,
    "I": 2,
    "E": 3
}

predictions = [
    id2label[int(t)]
    for t in pred_ids
    if int(t) != 0
]

In [ ]:
print(predictions[:100])

In [ ]:
import pandas as pd

sample["Predicted"] = predictions

sample.to_csv("/content/final_submission.csv", index=False)